# 비율 파생을 선형 멤버에 — 강화 로지스틱 + 블렌드 증분

**가설:** 비율(num/den)은 트리가 흡수하지만 **선형은 원리적으로 못 만들어**(가중합뿐) → 비율을 직접 주면 선형엔 *새 정보.* B3가 "트리가 흡수한 상호작용을 선형이 씀"을 입증했으나 *비율 세트는 안 줬음.*
**검증:** 같은 파이프라인으로 **비율 X vs 비율 O** 선형 멤버 → (1)단독 AUC가 강해지나 (2)계수로 비율 사용 확인 (3)트리와 ρ (4)**블렌드 증분**(이게 채택 관문).
**입력:** train/test + `oof_predictions.csv`(트리) + `oof_nn.csv`. 누수안전: fold-내부 TE·impute·scale.

## 0. 준비 + tf_tree(숫자 베이스) + 비율(v1+게이팅) + TE

In [1]:
import os,glob,re,time
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata
def find_csv(n):
    h=[p for p in glob.glob("/kaggle/input/**/*.csv",recursive=True) if os.path.basename(p)==n]; assert h,f"{n} 없음"; return sorted(h,key=len)[0]
train=pd.read_csv(find_csv("train.csv")); test=pd.read_csv(find_csv("test.csv"))
TARGET="임신 성공 여부"; ID_COL="ID"; y=train[TARGET].astype(int).values; N=len(train)
def NUM(df,c): return pd.to_numeric(df[c],errors="coerce") if c in df else pd.Series(np.nan,index=df.index)
def DIV(num,den): den=den.astype(float); return num.astype(float)/den.where(den>0)
def runr(x): return rankdata(x)/len(x)
COL_PROC="특정 시술 유형"; COL_RSN="배아 생성 주요 이유"
NOMINAL_COLS=["시술 시기 코드","시술 유형","배란 유도 유형","난자 출처","정자 출처"]
OCC=["총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수","DI 시술 횟수","총 임신 횟수","IVF 임신 횟수","DI 임신 횟수","총 출산 횟수","IVF 출산 횟수","DI 출산 횟수"]
AGE_MAPS={"시술 당시 나이":{"만18-34세":0,"만35-37세":1,"만38-39세":2,"만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":-1},
 "난자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1},
 "정자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1}}
CMAP={"0회":0,"1회":1,"2회":2,"3회":3,"4회":4,"5회":5,"6회 이상":6}
_tp=lambda s: [] if pd.isna(s) else [t.strip() for t in re.split(r"[/:]",str(s)) if t.strip()]
def fit_tree(tr):
    st={}; ig={TARGET,ID_COL}
    st["dead"]=[c for c in tr.columns if c not in ig and tr[c].nunique(dropna=True)<=1]
    st["sparse"]=[c for c in tr.columns if c not in ig and c not in st["dead"] and tr[c].isna().mean()>0.98]
    st["lc"]={c:pd.Index(tr[c].astype("category").cat.categories) for c in NOMINAL_COLS if c in tr}
    st["pv"]=sorted({t for L in tr[COL_PROC].apply(_tp) for t in L}); return st
def tf_tree(df,st):
    df=df.copy()
    if TARGET in df: df=df.drop(columns=[TARGET])
    df["is_DI"]=(df["시술 유형"]=="DI").astype(int)
    df=df.drop(columns=[c for c in st["dead"] if c in df.columns])
    for c in st["sparse"]:
        if c in df: df[f"{c}_있음"]=df[c].notna().astype(int); df=df.drop(columns=[c])
    for c in OCC:
        if c in df: df[c]=df[c].astype(object).map(CMAP)
    for c,m in AGE_MAPS.items():
        if c in df: df[c]=df[c].astype(object).map(m)
    cats=[]
    for c,cc in st["lc"].items():
        if c in df: df[c]=pd.Categorical(df[c],categories=cc).codes.astype("int32"); cats.append(c)
    ts=df[COL_PROC].apply(_tp)
    for v in st["pv"]: df[f"proc_{v}"]=ts.apply(lambda L,v=v:int(v in L))
    df=df.drop(columns=[c for c in [COL_PROC,COL_RSN,ID_COL] if c in df.columns])
    obj=[c for c in df.columns if df[c].dtype==object]
    if obj: df=df.drop(columns=obj)
    for c in cats: df[c]=df[c].fillna(-1).astype("int32")
    return df,[c for c in cats if c in df.columns]
# 비율 세트 (v1 풀링 + v2 게이팅)
def masks(df): return {"신선":NUM(df,"신선 배아 사용 여부")==1,"동결":NUM(df,"동결 배아 사용 여부")==1,"ICSI":NUM(df,"미세주입된 난자 수")>0,"본인난자":df["난자 출처"].astype(str)=="본인 제공","기증난자":df["난자 출처"].astype(str)=="기증 제공"}
def build_ratios(df):
    M=masks(df); F={}
    P1=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"혼합된 난자 수")); P2=DIV(NUM(df,"미세주입에서 생성된 배아 수"),NUM(df,"미세주입된 난자 수"))
    P6=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"수집된 신선 난자 수")); L3=NUM(df,"배아 이식 경과일")-NUM(df,"난자 혼합 경과일")
    F["P1_수정률"]=P1; F["P2_ICSI수정"]=P2; F["P6_난자수율"]=P6; F["L3_배양일수"]=L3
    F["P3_활용률"]=DIV(NUM(df,"이식된 배아 수")+NUM(df,"저장된 배아 수"),NUM(df,"총 생성 배아 수")); F["P5_저장률"]=DIV(NUM(df,"저장된 배아 수"),NUM(df,"총 생성 배아 수"))
    F["L6_배아per ICSI난자"]=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"미세주입된 난자 수")); F["N6_난자감모"]=DIV(NUM(df,"수집된 신선 난자 수")-NUM(df,"혼합된 난자 수"),NUM(df,"수집된 신선 난자 수"))
    # 게이팅
    F["g신선_수정률"]=P1.where(M["신선"]); F["gICSI_수정효율"]=P2.where(M["ICSI"]); F["g본인_수율"]=P6.where(M["본인난자"]); F["g기증_수율"]=P6.where(M["기증난자"]); F["g신선_배양일수"]=L3.where(M["신선"])
    F["FZ1_동결해동이식간격"]=(NUM(df,"배아 이식 경과일")-NUM(df,"배아 해동 경과일")).where(M["동결"]); F["FZ2_해동이식률"]=DIV(NUM(df,"이식된 배아 수"),NUM(df,"해동된 배아 수"))
    return pd.DataFrame(F,index=df.index)
st=fit_tree(train); Xb,CATF=tf_tree(train,st); Xb_te,_=tf_tree(test,st); Xb_te=Xb_te.reindex(columns=Xb.columns)
base_num=Xb.drop(columns=CATF); base_num_te=Xb_te.drop(columns=CATF)
RAT=build_ratios(train); RAT_te=build_ratios(test)
TE_COLS=NOMINAL_COLS+[COL_PROC,COL_RSN]
def te_fit(cat,yy,sm=20):
    g=pd.DataFrame({"c":cat.values,"y":yy}).groupby("c")["y"].agg(["mean","count"]); pr=float(yy.mean())
    return ((g["mean"]*g["count"]+pr*sm)/(g["count"]+sm)), pr
print("선형 숫자베이스",base_num.shape[1],"| 비율",RAT.shape[1],"| TE대상",len(TE_COLS))

선형 숫자베이스 67 | 비율 15 | TE대상 7


## 1. 강화 선형 멤버 OOF — 비율 X vs 비율 O (fold-내부 TE·impute·scale)

In [2]:
def lin_oof(use_ratios,seed=42,full_test=False):
    oof=np.zeros(N); tst=np.zeros(len(test)) if full_test else None
    for tri,vai in StratifiedKFold(5,shuffle=True,random_state=seed).split(base_num,y):
        Xt=base_num.iloc[tri].copy(); Xv=base_num.iloc[vai].copy(); Xte=base_num_te.copy() if full_test else None
        for c in TE_COLS:
            enc,pr=te_fit(train[c].astype(str).iloc[tri],y[tri])
            Xt[f"te_{c}"]=train[c].astype(str).iloc[tri].map(enc).fillna(pr).values
            Xv[f"te_{c}"]=train[c].astype(str).iloc[vai].map(enc).fillna(pr).values
            if full_test: Xte[f"te_{c}"]=test[c].astype(str).map(enc).fillna(pr).values
        if use_ratios:
            Xt=pd.concat([Xt.reset_index(drop=True),RAT.iloc[tri].reset_index(drop=True)],axis=1)
            Xv=pd.concat([Xv.reset_index(drop=True),RAT.iloc[vai].reset_index(drop=True)],axis=1)
            if full_test: Xte=pd.concat([Xte.reset_index(drop=True),RAT_te.reset_index(drop=True)],axis=1)
        med=Xt.median(); Xt=Xt.fillna(med); Xv=Xv.fillna(med)
        sc=StandardScaler().fit(Xt); m=LogisticRegression(max_iter=2000,C=0.5).fit(sc.transform(Xt),y[tri])
        oof[vai]=m.predict_proba(sc.transform(Xv))[:,1]
        if full_test: tst+=m.predict_proba(sc.transform(Xte.fillna(med)))[:,1]/5
    return (oof,tst) if full_test else oof
t0=time.time(); lin0=lin_oof(False); lin1=lin_oof(True)
a0,a1=roc_auc_score(y,lin0),roc_auc_score(y,lin1)
print(f"선형 (비율 X): OOF={a0:.5f}")
print(f"선형 (비율 O): OOF={a1:.5f}   Δ={a1-a0:+.5f}  ← 비율이 선형을 강하게?  ({time.time()-t0:.0f}s)")
print(f"B3 선형(0.719) 대비: {a1-0.719:+.4f}")
print("ρ(비율X, 비율O):",round(np.corrcoef(runr(lin0),runr(lin1))[0,1],3))

선형 (비율 X): OOF=0.71507
선형 (비율 O): OOF=0.72029   Δ=+0.00521  ← 비율이 선형을 강하게?  (45s)
B3 선형(0.719) 대비: +0.0013
ρ(비율X, 비율O): 0.979


## 2. 어떤 비율을 선형이 쓰나 — 표준화 계수 (full fit, 진단용)

In [3]:
Xf=base_num.copy()
for c in TE_COLS:
    enc,pr=te_fit(train[c].astype(str),y); Xf[f"te_{c}"]=train[c].astype(str).map(enc).fillna(pr).values
Xf=pd.concat([Xf.reset_index(drop=True),RAT.reset_index(drop=True)],axis=1); Xf=Xf.fillna(Xf.median())
sc=StandardScaler().fit(Xf); mf=LogisticRegression(max_iter=2000,C=0.5).fit(sc.transform(Xf),y)
coef=pd.Series(mf.coef_[0],index=Xf.columns)
print("비율 피처 표준화 계수 (|값| 큰 순 — 선형이 실제로 쓰는 비율):")
print(coef[RAT.columns].reindex(coef[RAT.columns].abs().sort_values(ascending=False).index).head(12).round(3).to_string())

비율 피처 표준화 계수 (|값| 큰 순 — 선형이 실제로 쓰는 비율):
P5_저장률             0.283
L3_배양일수            0.261
g신선_배양일수           0.226
FZ1_동결해동이식간격       0.140
FZ2_해동이식률          0.095
N6_난자감모           -0.071
g신선_수정률            0.040
P3_활용률            -0.024
gICSI_수정효율         0.009
P2_ICSI수정          0.009
L6_배아per ICSI난자    0.008
P6_난자수율           -0.007


## 3. lr·nn·트리 OOF 로드 (버그수정·하드스톱)

In [4]:
def load():
    tr={}; nn=None
    for p in glob.glob("/kaggle/input/**/oof_predictions.csv",recursive=True):
        d=pd.read_csv(p)
        for k in d.columns:
            kl=k.lower()
            if kl in("oof_lgb","oof_cat","oof_xgb") or kl.replace("oof_","") in ("lgb","cat","xgb"): tr[kl.replace("oof_","")]=d[k].values
        break
    for p in glob.glob("/kaggle/input/**/oof_nn.csv",recursive=True):
        d=pd.read_csv(p); k=[c for c in d.columns if c.lower()=="oof_nn"] or [c for c in d.columns if "nn" in c.lower()]; nn=d[k[0]].values if k else None; break
    return tr,nn
trees,oof_nn=load()
assert len(trees)>=2 and oof_nn is not None, "트리/nn OOF 부족 — oof_predictions.csv·oof_nn.csv Add Input 필수"
assert roc_auc_score(y,oof_nn)<0.999, "nn 라벨 잘못 로드 의심"
print("로드 트리:",{k:round(roc_auc_score(y,v),5) for k,v in trees.items()},"| nn",round(roc_auc_score(y,oof_nn),5))

로드 트리: {'lgb': np.float64(0.73944), 'cat': np.float64(0.73963), 'xgb': np.float64(0.73959)} | nn 0.73751


## 4. 블렌드 증분 — 트리+nn에 (비율X 선형) vs (비율O 선형)

In [5]:
def hill(d,yy,n=120):
    nm=list(d); s0={k:roc_auc_score(yy,d[k]) for k in nm}; b=max(s0,key=s0.get); ens=[b]; s=d[b].copy(); best=(list(ens),s0[b])
    for _ in range(n):
        cb,ca=None,-1
        for k in nm:
            a=roc_auc_score(yy,(s+d[k])/(len(ens)+1))
            if a>ca: ca,cb=a,k
        ens.append(cb); s=s+d[cb]
        if ca>best[1]: best=(list(ens),ca)
    from collections import Counter; c=Counter(best[0]); return {k:c.get(k,0)/len(best[0]) for k in nm}, best[1]
def blend(lin):
    d={f"t_{k}":runr(trees[k]) for k in trees}; d["nn"]=runr(oof_nn); d["lin"]=runr(lin)
    w,a=hill(d,y); return d,w,a
dB,wB,aB=blend(lin0); dF,wF,aF=blend(lin1)
print(f"블렌드 (비율X 선형) OOF={aB:.5f} weights={ {k:round(v,3) for k,v in wB.items() if v>0} }")
print(f"블렌드 (비율O 선형) OOF={aF:.5f} weights={ {k:round(v,3) for k,v in wF.items() if v>0} }")
print(f"증분 Δ = {aF-aB:+.5f}")
pB=sum(wB[k]*dB[k] for k in wB); pF=sum(wF[k]*dF[k] for k in wF)
rng=np.random.default_rng(0); ds=[roc_auc_score(y[ix],pF[ix])-roc_auc_score(y[ix],pB[ix]) for ix in (rng.integers(0,N,N) for _ in range(2000))]
lo,hi=np.percentile(ds,[2.5,97.5]); gate=lo>0
print(f"\n블렌드 증분(비율효과) = {aF-aB:+.5f} | 95% CI [{lo:+.5f},{hi:+.5f}]")
print("게이트(CI>0):","✅ 비율이 선형 통해 블렌드 기여 — 멀티시드+LB" if gate else "❌ 미달")
print("\n참고: lin 단독 ρ(트리max):",round(max(np.corrcoef(runr(lin1),runr(trees[k]))[0,1] for k in trees),3),"(<0.93면 decorrelated)")

블렌드 (비율X 선형) OOF=0.74045 weights={'t_lgb': 0.209, 't_cat': 0.275, 't_xgb': 0.253, 'nn': 0.242, 'lin': 0.022}
블렌드 (비율O 선형) OOF=0.74046 weights={'t_lgb': 0.206, 't_cat': 0.27, 't_xgb': 0.254, 'nn': 0.238, 'lin': 0.032}
증분 Δ = +0.00001

블렌드 증분(비율효과) = +0.00001 | 95% CI [-0.00000,+0.00003]
게이트(CI>0): ❌ 미달

참고: lin 단독 ρ(트리max): 0.903 (<0.93면 decorrelated)


## 5. 멀티시드(게이트 통과 시) + 출력

In [6]:
import json
inc=[aF-aB]
if gate:
    print("게이트 통과 → seed 7/2024 증분:")
    for sd in (7,2024):
        l0=lin_oof(False,sd); l1=lin_oof(True,sd)
        _,wb,ab=blend(l0); _,wf,af=blend(l1); inc.append(af-ab); print(f"  seed{sd}: 증분={af-ab:+.5f}")
    mu,s=np.mean(inc),np.std(inc); print(f"멀티시드 증분 {mu:+.5f}±{s:.5f} →","✅ 채택·LB" if mu>0 and abs(mu)>s else "🟡 시드 불안정")
else: print("게이트 미달 → 멀티시드 불필요.")
# 비율O 선형 OOF/test 저장(채택 시 운영 블렌드 재사용)
_,lin1_te=lin_oof(True,42,full_test=True)
pd.DataFrame({"oof_lin_ratio":lin1,"y":y}).to_csv("/kaggle/working/oof_lin_ratio.csv",index=False)
pd.DataFrame({"ID":test[ID_COL] if ID_COL in test else np.arange(len(test)),"lin_ratio":lin1_te}).to_csv("/kaggle/working/test_lin_ratio.csv",index=False)
json.dump({"lin_noratio":a0,"lin_ratio":a1,"blend_inc":aF-aB,"ci":[lo,hi],"gate":bool(gate)},open("/kaggle/working/linear_decision.json","w"),ensure_ascii=False)
print("💾 oof_lin_ratio.csv / test_lin_ratio.csv / linear_decision.json 저장")

게이트 미달 → 멀티시드 불필요.
💾 oof_lin_ratio.csv / test_lin_ratio.csv / linear_decision.json 저장
